In [1]:
import os
import glob
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
import timm
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

Menggunakan device: cuda:0


### Load Kedua Model & Transformasi Khusus

In [2]:
class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']

# 1. Transformasi untuk EfficientNet-B3 (300x300)
transform_eff = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Transformasi untuk ConvNeXt-Tiny (224x224)
transform_conv = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Inisialisasi MTCNN
mtcnn = MTCNN(keep_all=False, select_largest=True, margin=60, post_process=False, device=device)

# 4. Load Model 1 (EfficientNet-B3)
model_eff = timm.create_model('efficientnet_b3', pretrained=False, num_classes=len(class_names))
model_eff.load_state_dict(torch.load('../models/efficientnet_b3_model.pth', map_location=device))
model_eff = model_eff.to(device)
model_eff.eval()

# 5. Load Model 2 (ConvNeXt-Tiny)
model_conv = timm.create_model('convnext_tiny', pretrained=False, num_classes=len(class_names))
model_conv.load_state_dict(torch.load('../models/convnext_tiny_model.pth', map_location=device))
model_conv = model_conv.to(device)
model_conv.eval()

print("Kedua Model, MTCNN, dan Transformasi siap berkolaborasi!")

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_pat

Kedua Model, MTCNN, dan Transformasi siap berkolaborasi!


### Proses Inference Ensembling (Keroyokan)

In [3]:
test_dir = '../Data/test/' 
submission_df = pd.read_csv('../Data/samplesubmission.csv')
predictions = []

print("Memulai proses Prediksi Ensemble (B3 + ConvNeXt)")
with torch.no_grad():
    for img_id in tqdm(submission_df['id']):
        search_pattern = os.path.join(test_dir, f"{img_id}.*")
        matching_files = glob.glob(search_pattern)
        
        if not matching_files:
            predictions.append('fake_unknown')
            continue
            
        img_path = matching_files[0] 
        
        try:
            img = Image.open(img_path).convert('RGB')
            face = mtcnn(img)
            
            if face is not None:
                pil_img = transforms.ToPILImage()(face.byte())
            else:
                pil_img = img 
            
            # SIAPKAN TENSOR UNTUK MASING-MASING MODEL 
            tensor_eff = transform_eff(pil_img).unsqueeze(0).to(device)
            tensor_conv = transform_conv(pil_img).unsqueeze(0).to(device)
            
            # TEBAK BERSAMA 
            output_eff = model_eff(tensor_eff)
            output_conv = model_conv(tensor_conv)
            
            # Ubah ke probabilitas persentase (Softmax)
            prob_eff = F.softmax(output_eff, dim=1)
            prob_conv = F.softmax(output_conv, dim=1)
            
            # Hitung rata-rata probabilitas keduanya (VOTING)

            # avg_prob = (prob_eff + prob_conv) / 2.0
            avg_prob = (prob_eff * 0.35) + (prob_conv * 0.65)
            
            # Ambil nilai tertinggi dari hasil rata-rata
            _, predicted_idx = torch.max(avg_prob, 1)
            predicted_class = class_names[predicted_idx.item()]
            
            predictions.append(predicted_class)
            
        except Exception as e:
            print(f"Error memproses {img_path}: {e}")
            predictions.append('fake_unknown')

Memulai proses Prediksi Ensemble (B3 + ConvNeXt)


  0%|          | 0/404 [00:00<?, ?it/s]

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [5]:
import glob
import torch.nn.functional as F

test_dir = '../Data/test/' 
submission_df = pd.read_csv('../Data/samplesubmission.csv')
predictions = []

# Buat 3 detektor wajah dengan tingkat zoom (margin) yang berbeda-beda
mtcnn_tight = MTCNN(keep_all=False, select_largest=True, margin=20, post_process=False, device=device)
mtcnn_normal = MTCNN(keep_all=False, select_largest=True, margin=60, post_process=False, device=device)
mtcnn_wide = MTCNN(keep_all=False, select_largest=True, margin=100, post_process=False, device=device)

print("🚀 Memulai Prediksi Ultimate Ensemble (Multi-Zoom B3 + ConvNeXt)...")
with torch.no_grad():
    for img_id in tqdm(submission_df['id']):
        search_pattern = os.path.join(test_dir, f"{img_id}.*")
        matching_files = glob.glob(search_pattern)
        
        if not matching_files:
            predictions.append('fake_unknown')
            continue
            
        img_path = matching_files[0] 
        
        try:
            img = Image.open(img_path).convert('RGB')
            
            # --- 1. AMBIL 3 VERSI WAJAH ---
            faces = [mtcnn_tight(img), mtcnn_normal(img), mtcnn_wide(img)]
            
            all_probs_eff = []
            all_probs_conv = []
            
            for i, face in enumerate(faces):
                if face is not None:
                    pil_img = transforms.ToPILImage()(face.byte())
                else:
                    # Smart Center Crop Fallback menyesuaikan level zoom
                    w, h = img.size
                    # Zoom dekat: ambil 40% area tengah. Normal: 60%. Jauh: 80%.
                    ratio = 0.4 if i == 0 else (0.6 if i == 1 else 0.8)
                    margin = (1.0 - ratio) / 2.0
                    
                    left = w * margin
                    top = h * margin
                    right = w * (1.0 - margin)
                    bottom = h * (1.0 - margin)
                    
                    pil_img = img.crop((left, top, right, bottom))
                    
                # Siapkan Tensor untuk masing-masing model
                tensor_eff = transform_eff(pil_img).unsqueeze(0).to(device)
                tensor_conv = transform_conv(pil_img).unsqueeze(0).to(device)
                
                # Tebak!
                output_eff = model_eff(tensor_eff)
                output_conv = model_conv(tensor_conv)
                
                # Simpan probabilitasnya (bentuk persentase)
                all_probs_eff.append(F.softmax(output_eff, dim=1))
                all_probs_conv.append(F.softmax(output_conv, dim=1))
            
            # --- 2. VOTING TINGKAT TINGGI ---
            # Rata-ratakan hasil dari 3 level zoom untuk masing-masing model
            avg_zoom_eff = torch.mean(torch.stack(all_probs_eff), dim=0)
            avg_zoom_conv = torch.mean(torch.stack(all_probs_conv), dim=0)
            
            # Weighted Voting (ConvNeXt lebih cerdas, beri porsi 65%)
            final_prob = (avg_zoom_eff * 0.35) + (avg_zoom_conv * 0.65)
            
            # Ambil kelas pemenang akhir
            _, predicted_idx = torch.max(final_prob, 1)
            predicted_class = class_names[predicted_idx.item()]
            
            predictions.append(predicted_class)
            
        except Exception as e:
            print(f"❌ Error memproses {img_path}: {e}")
            predictions.append('fake_unknown')

print("Proses Prediksi Ultimate Selesai!")

🚀 Memulai Prediksi Ultimate Ensemble (Multi-Zoom B3 + ConvNeXt)...


d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_pat

  0%|          | 0/404 [00:00<?, ?it/s]

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Proses Prediksi Ultimate Selesai!


In [ ]:
import os

submission_dir = '../Data/submission/'
os.makedirs(submission_dir, exist_ok=True)

submission_df['label'] = predictions

# Simpan dengan nama ensemble
file_name = 'ubmission_ensemble_b3_convnext_multizoom_hafisDatase.csv'
submission_file_path = os.path.join(submission_dir, file_name)

submission_df.to_csv(submission_file_path, index=False)

print("="*40)
print(f"File submission Ensemble berhasil disimpan!")
print(f"Lokasi: {submission_file_path}")
print("="*40)
display(submission_df.head(10))

File submission Ensemble berhasil disimpan!
Lokasi: ../Data/submission/ubmission_ensemble_b3_convnext_multizoom.csv


,id,label
0,test_001,fake_printed
1,test_002,fake_screen
2,test_003,fake_mask
3,test_004,realperson
4,test_005,realperson
5,test_006,fake_unknown
6,test_007,fake_mask
7,test_008,fake_mannequin
8,test_009,fake_mask
9,test_010,fake_screen


In [7]:
# Melihat seberapa sering model menebak masing-masing kelas
distribusi = submission_df['label'].value_counts()
print("Distribusi Tebakan Model pada Data Test:")
print(distribusi)

# Menampilkan dalam persentase
print("\nPersentase:")
print((distribusi / len(submission_df)) * 100)

Distribusi Tebakan Model pada Data Test:
label
realperson        108
fake_mask          70
fake_printed       64
fake_screen        60
fake_unknown       51
fake_mannequin     51
Name: count, dtype: int64

Persentase:
label
realperson        26.732673
fake_mask         17.326733
fake_printed      15.841584
fake_screen       14.851485
fake_unknown      12.623762
fake_mannequin    12.623762
Name: count, dtype: float64
